In [1]:
!pip install -q transformers datasets evaluate seqeval matplotlib seaborn scikit-learn pytorch-crf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [2]:
# Load library
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from torchcrf import CRF
from IPython.display import display

In [3]:
# config
CHOSEN_MODEL = 'distibert-base-cased'
DATASET_FILE = '/content/dataset.jsonl'

RUN_MODE = 'ALL'

print(f"Model type: {CHOSEN_MODEL.upper()}")
print(f"Run mode: {RUN_MODE}")

Model type: DISTIBERT-BASE-CASED
Run mode: ALL


In [6]:
# load and prepare data
dataset = load_dataset('json', data_files=DATASET_FILE, split='train')
dataset = dataset.train_test_split(test_size=0.2, seed=42)

all_tags = [tag for tags_list in dataset["train"]["ner_tags"] for tag in tags_list]
unique_tags = sorted(list(set(all_tags)))

label2id = {label: i for i, label in enumerate(unique_tags)}
id2label = {i: label for i, label in enumerate(unique_tags)}

def encode_tags(example):
  example["ner_tags_id"] = [label2id.get(tag, 0) for tag in example["ner_tags"]]
  return example

dataset = dataset.map(encode_tags)

# cal class weight for focal loss
tag_counts = pd.Series(all_tags).value_counts()
total_tags = len(all_tags)
class_weights = [total_tags / (len(unique_tags) * tag_counts[label]) for label in unique_tags]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)